# Numba Acceleration with Python

## Motivation

Python is widely used in scientific computing, data science, and high-performance computing due to its simplicity and extensive ecosystem. However, pure Python loops are generally slow because Python is an interpreted language with significant runtime overhead.

NumPy improves performance by providing highly optimized vectorized operations implemented in low-level languages such as C and Fortran. Nevertheless, NumPy alone is sometimes insufficient for:

- Complex loop-based algorithms
- Custom reductions
- Stencil computations
- Monte Carlo simulations
- Irregular memory access patterns
- Parallel workloads

This is where Numba becomes useful.

---

# What is Numba?

Numba is a Just-In-Time (JIT) compiler for Python focused on numerical computing.

It translates a subset of Python and NumPy code into optimized machine code at runtime using LLVM.

With minimal code modifications, Numba can provide:

- Native machine code execution
- Automatic vectorization
- Multi-threaded parallelism
- Reduced Python overhead
- Significant performance improvements

---

# Core Numba Concepts

## `@jit` and `@njit`

Numba accelerates functions using decorators.

Example:

```python
@nb.jit
def f(x):
    ...
```

or

```python
@nb.njit
def f(x):
    ...
```

`njit` is equivalent to:

```python
@nb.jit(nopython=True)
```

and is generally preferred for HPC workloads.

---

# Nopython Mode

```python
nopython=True
```

This forces Numba to compile the function entirely to native code without falling back to the Python interpreter.

This is critical for obtaining high performance.

---

# Parallel Execution

```python
parallel=True
```

Enables automatic loop parallelization.

Combined with:

```python
nb.prange(...)
```

Numba distributes loop iterations across multiple CPU threads.

Example:

```python
for i in nb.prange(N):
    ...
```

This is conceptually similar to OpenMP parallel loops in C/C++.

---

# Fast Math Optimizations

```python
fastmath=True
```

Allows aggressive floating-point optimizations such as:

- reassociation,
- vectorization,
- fused operations.

This can improve performance substantially, although it may slightly reduce strict IEEE floating-point reproducibility.

---

# No GIL

```python
nogil=True
```

Releases Python's Global Interpreter Lock (GIL) during execution.

This allows true multithreaded execution.

---

# Explicit Signatures

Numba can infer types automatically, but we may also provide explicit type signatures.

Example:

```python
@nb.jit(nb.float64(nb.float64[:], nb.float64))
```

Advantages:
- avoids runtime type inference,
- prevents recompilation,
- reduces startup overhead,
- improves reproducibility.

Runtime performance is usually similar after compilation.

---

# Warmup Compilation

Numba uses Just-In-Time compilation.

The first execution includes:
1. type inference,
2. LLVM compilation,
3. machine code generation.

Therefore, the first call is not representative of actual performance.

We perform a warmup execution before benchmarking to exclude compilation overhead.

---

# Benchmarking Methodology

For each implementation we will:
1. warm up the kernel,
2. execute multiple runs,
3. measure execution time,
4. compute average and best timings.

This provides more reliable performance measurements.

---

The implementations of the examples will be:

## 1. Pure NumPy

Uses vectorized array operations.

Purpose:
- establish a high-performance baseline,
- demonstrate optimized vectorization.

---

## 2. Numba without Explicit Signature

Uses:
- JIT compilation,
- parallel loops,
- native machine code generation.

Purpose:
- demonstrate loop acceleration,
- show multithreaded execution,
- compare against NumPy.

---

## 3. Numba with Explicit Signature

Same kernel as Example 2 but with explicit typing.

Purpose:
- demonstrate signature specialization,
- discuss compilation overhead,
- compare runtime behavior.

---

# Performance Questions

During the experiments we will analyze:

- How much acceleration does Numba provide?
- Is NumPy already optimal?
- Does parallelization improve performance?
- What is the impact of explicit signatures?
- What is the effect of JIT compilation overhead?
- How important are memory bandwidth and reductions?

---

# Important Observation

High-performance computing optimization is rarely about a single trick.

Performance depends on:
- algorithmic complexity,
- vectorization,
- memory access patterns,
- threading,
- cache efficiency,
- compiler optimizations,
- workload size.

Numba provides an accessible framework for exploring these concepts directly in Python.

# 1. $\pi$ approximation
1. Approximate $\pi$ calculation:
$$
\pi = \int_0^1 \frac{4}{1+x^2} dx \approx \sum_k \frac{4}{1+x^2}\Delta x
$$

<center>
  <img src="https://dotink.co/img/riemann-sum.jpg" width="500" />
</center>

In [8]:
import numpy as np
import numba as nb
import time

# ============================================================
# BENCHMARK FUNCTION UTILITY
# ============================================================

def benchmark(func, *args, repeat=5):

    times = []

    result = None

    for i in range(repeat):

        t0 = time.perf_counter()

        result = func(*args)

        t1 = time.perf_counter()

        elapsed = t1 - t0

        times.append(elapsed)

        print(f"Run {i+1}: {elapsed:.6f} s")

    return result, np.mean(times), np.min(times)

In [5]:
N = 125_000_000

dx = 1.0 / N
x = np.arange(dx/2, 1 + dx, dx)

pi = 3.141592653589793

# ============================================================
# VERSION 1: PURE NUMPY
# ============================================================

def get_pi_numpy(x, dx):

    s = np.sum(dx * 4.0 / (1.0 + x**2))

    return s

# ============================================================
# VERSION 2: NUMBA WITHOUT SIGNATURE
# ============================================================

@nb.jit(
    nopython=True,
    parallel=True,
    nogil=True,
    fastmath=True,
    boundscheck=False,
    cache=True
)
def get_pi_numba(x, dx):

    s = 0.0

    for i in nb.prange(x.shape[0]):
        s += dx * 4.0 / (1.0 + x[i]**2)

    return s

# ============================================================
# VERSION 3: NUMBA WITH SIGNATURE
# ============================================================

@nb.jit(
    nb.float64(nb.float64[:], nb.float64),
    nopython=True,
    parallel=True,
    nogil=True,
    fastmath=True,
    boundscheck=False,
    cache=True
)
def get_pi_numba_signature(x, dx):

    s = 0.0

    for i in nb.prange(x.shape[0]):
        s += dx * 4.0 / (1.0 + x[i]**2)

    return s

# ============================================================
# WARMUP
# ============================================================

print("Running warmup compilation...\n")

get_pi_numba(x[:1000], dx)
get_pi_numba_signature(x[:1000], dx)

# ============================================================
# NUMPY
# ============================================================

print("============== NUMPY ==============")

pi_numpy, numpy_avg, numpy_best = benchmark(
    get_pi_numpy,
    x,
    dx
)

# ============================================================
# NUMBA
# ============================================================

print("\n============== NUMBA ==============")

pi_numba, numba_avg, numba_best = benchmark(
    get_pi_numba,
    x,
    dx
)

# ============================================================
# NUMBA SIGNATURE
# ============================================================

print("\n========= NUMBA SIGNATURE =========")

pi_numba_sig, sig_avg, sig_best = benchmark(
    get_pi_numba_signature,
    x,
    dx
)

# ============================================================
# RESULTS
# ============================================================

print("\n================ FINAL RESULTS ================\n")

print(f"Reference PI          : {pi}\n")

print("NumPy")
print(f"  Estimate            : {pi_numpy}")
print(f"  Error               : {abs(pi_numpy - pi)}")
print(f"  Average Time        : {numpy_avg:.6f} s")
print(f"  Best Time           : {numpy_best:.6f} s\n")

print("Numba")
print(f"  Estimate            : {pi_numba}")
print(f"  Error               : {abs(pi_numba - pi)}")
print(f"  Average Time        : {numba_avg:.6f} s")
print(f"  Best Time           : {numba_best:.6f} s\n")

print("Numba + Signature")
print(f"  Estimate            : {pi_numba_sig}")
print(f"  Error               : {abs(pi_numba_sig - pi)}")
print(f"  Average Time        : {sig_avg:.6f} s")
print(f"  Best Time           : {sig_best:.6f} s\n")

Running warmup compilation...

============== NUMPY ==============
Run 1: 4.138875 s
Run 2: 2.097429 s
Run 3: 0.844717 s
Run 4: 0.994189 s
Run 5: 0.798759 s

============== NUMBA ==============
Run 1: 0.039085 s
Run 2: 0.059184 s
Run 3: 0.058925 s
Run 4: 0.059607 s
Run 5: 0.051186 s

========= NUMBA SIGNATURE =========
Run 1: 0.042265 s
Run 2: 0.054296 s
Run 3: 0.036952 s
Run 4: 0.056240 s
Run 5: 0.039142 s

================ FINAL RESULTS ================

Reference PI          : 3.141592653589793

NumPy
  Estimate            : 3.141592669589793
  Error               : 1.5999999991578306e-08
  Average Time        : 1.774794 s
  Best Time           : 0.798759 s

Numba
  Estimate            : 3.1415926695897944
  Error               : 1.6000001323845936e-08
  Average Time        : 0.053597 s
  Best Time           : 0.039085 s

Numba + Signature
  Estimate            : 3.1415926695897944
  Error               : 1.6000001323845936e-08
  Average Time        : 0.045779 s
  Best Time         

# 2. Finite Difference Stencil Kernel

## Motivation

Stencil computations are among the most important workloads in scientific computing and high-performance computing.

They appear in:
- Computational Fluid Dynamics (CFD)
- Heat transfer
- Shallow Water Equations
- Wave propagation
- Image processing
- Finite Difference Methods (FDM)
- Finite Volume Methods (FVM)

A stencil operation computes new values using neighboring elements in an array or grid.

---

# The Laplacian Operator

In this example we compute the 1D discrete Laplacian:

$$
u_{i-1} - 2u_i + u_{i+1}
$$

This is the second-order finite difference approximation of:

$$
\frac{\partial^2 u}{\partial x^2}
$$

and is fundamental in many PDE solvers.

---

In [10]:
# ============================================================
# EXAMPLE 2: FINITE DIFFERENCE / STENCIL KERNEL
# ============================================================

import numpy as np
import numba as nb
import time

N = 50_000_000

u = np.random.rand(N)


# ============================================================
# NUMPY VERSION
# ============================================================

def laplacian_numpy(u):

    return u[:-2] - 2.0*u[1:-1] + u[2:]


# ============================================================
# NUMBA VERSION
# ============================================================

@nb.jit(
    nopython=True,
    parallel=True,
    nogil=True,
    fastmath=True,
    boundscheck=False,
    cache=True
)
def laplacian_numba(u):

    out = np.empty(u.shape[0] - 2)

    for i in nb.prange(1, u.shape[0] - 1):

        out[i - 1] = u[i - 1] - 2.0*u[i] + u[i + 1]

    return out


# ============================================================
# NUMBA SIGNATURE VERSION
# ============================================================

@nb.jit(
    nb.float64[:](nb.float64[:]),
    nopython=True,
    parallel=True,
    nogil=True,
    fastmath=True,
    boundscheck=False,
    cache=True
)
def laplacian_numba_signature(u):

    out = np.empty(u.shape[0] - 2)

    for i in nb.prange(1, u.shape[0] - 1):

        out[i - 1] = u[i - 1] - 2.0*u[i] + u[i + 1]

    return out


# ============================================================
# WARMUP
# ============================================================

print("Running warmup compilation...\n")

laplacian_numba(u[:1000])
laplacian_numba_signature(u[:1000])


# ============================================================
# BENCHMARK FUNCTION
# ============================================================

def benchmark(func, *args, repeat=5):

    times = []

    result = None

    for i in range(repeat):

        t0 = time.perf_counter()

        result = func(*args)

        t1 = time.perf_counter()

        elapsed = t1 - t0

        times.append(elapsed)

        print(f"Run {i+1}: {elapsed:.6f} s")

    return result, np.mean(times), np.min(times)


# ============================================================
# NUMPY
# ============================================================

print("============== NUMPY ==============")

a, numpy_avg, numpy_best = benchmark(
    laplacian_numpy,
    u
)


# ============================================================
# NUMBA
# ============================================================

print("\n============== NUMBA ==============")

b, numba_avg, numba_best = benchmark(
    laplacian_numba,
    u
)


# ============================================================
# NUMBA SIGNATURE
# ============================================================

print("\n========= NUMBA SIGNATURE =========")

c, sig_avg, sig_best = benchmark(
    laplacian_numba_signature,
    u
)


# ============================================================
# RESULTS
# ============================================================

print("\n================ FINAL RESULTS ================\n")

print(f"Max Difference NumPy vs Numba          : {np.max(np.abs(a - b))}")
print(f"Max Difference NumPy vs Signature      : {np.max(np.abs(a - c))}\n")

print("NumPy")
print(f"  Average Time : {numpy_avg:.6f} s")
print(f"  Best Time    : {numpy_best:.6f} s\n")

print("Numba")
print(f"  Average Time : {numba_avg:.6f} s")
print(f"  Best Time    : {numba_best:.6f} s\n")

print("Numba + Signature")
print(f"  Average Time : {sig_avg:.6f} s")
print(f"  Best Time    : {sig_best:.6f} s\n")

Running warmup compilation...

============== NUMPY ==============
Run 1: 0.657903 s
Run 2: 0.731513 s
Run 3: 0.330631 s
Run 4: 0.564907 s
Run 5: 0.379799 s

============== NUMBA ==============
Run 1: 0.052268 s
Run 2: 0.189368 s
Run 3: 0.061447 s
Run 4: 0.085070 s
Run 5: 0.065972 s

========= NUMBA SIGNATURE =========
Run 1: 0.115406 s
Run 2: 0.088403 s
Run 3: 0.068886 s
Run 4: 0.088885 s
Run 5: 0.061139 s

================ FINAL RESULTS ================

Max Difference NumPy vs Numba          : 0.0
Max Difference NumPy vs Signature      : 0.0

NumPy
  Average Time : 0.532951 s
  Best Time    : 0.330631 s

Numba
  Average Time : 0.090825 s
  Best Time    : 0.052268 s

Numba + Signature
  Average Time : 0.084544 s
  Best Time    : 0.061139 s



# 3. Monte Carlo Pi Estimation

## Motivation

Monte Carlo methods estimate numerical quantities using random sampling.

They are widely used in:
- physics,
- quantitative finance,
- uncertainty quantification,
- Bayesian statistics,
- particle simulations,
- radiation transport,
- machine learning.

Monte Carlo algorithms are especially important in HPC because they are often embarrassingly parallel.

---

# Estimating Pi

We estimate:

$$
\pi
$$

using random points inside the unit square.

A point:

$$
(x, y)
$$

belongs to the unit circle if:

$$
x^2 + y^2 \leq 1
$$

The ratio between:
- points inside the circle,
- total sampled points,

approximates:

$$
\frac{\pi}{4}
$$

Therefore:

$$
\pi \approx 4 \times \frac{\text{inside points}}{\text{total points}}
$$

---

In [11]:
# ============================================================
# EXAMPLE 3: MONTE CARLO PI ESTIMATION
# ============================================================

import numpy as np
import numba as nb
import time

N = 100_000_000


# ============================================================
# NUMPY VERSION
# ============================================================

def montecarlo_numpy(N):

    x = np.random.random(N)
    y = np.random.random(N)

    inside = np.sum(x*x + y*y <= 1.0)

    return 4.0 * inside / N


# ============================================================
# NUMBA VERSION
# ============================================================

@nb.jit(
    nopython=True,
    parallel=True,
    nogil=True,
    fastmath=True,
    boundscheck=False,
    cache=True
)
def montecarlo_numba(N):

    x = np.random.random(N)
    y = np.random.random(N)

    count = 0

    for i in nb.prange(N):

        if x[i]*x[i] + y[i]*y[i] <= 1.0:
            count += 1

    return 4.0 * count / N


# ============================================================
# NUMBA SIGNATURE VERSION
# ============================================================

@nb.jit(
    nb.float64(nb.int64),
    nopython=True,
    parallel=True,
    nogil=True,
    fastmath=True,
    boundscheck=False,
    cache=True
)
def montecarlo_numba_signature(N):

    x = np.random.random(N)
    y = np.random.random(N)

    count = 0

    for i in nb.prange(N):

        if x[i]*x[i] + y[i]*y[i] <= 1.0:
            count += 1

    return 4.0 * count / N


# ============================================================
# WARMUP
# ============================================================

print("Running warmup compilation...\n")

montecarlo_numba(1000)
montecarlo_numba_signature(1000)


# ============================================================
# BENCHMARK FUNCTION
# ============================================================

def benchmark(func, *args, repeat=5):

    times = []

    result = None

    for i in range(repeat):

        t0 = time.perf_counter()

        result = func(*args)

        t1 = time.perf_counter()

        elapsed = t1 - t0

        times.append(elapsed)

        print(f"Run {i+1}: {elapsed:.6f} s")

    return result, np.mean(times), np.min(times)


# ============================================================
# NUMPY
# ============================================================

print("============== NUMPY ==============")

pi_numpy, numpy_avg, numpy_best = benchmark(
    montecarlo_numpy,
    N
)


# ============================================================
# NUMBA
# ============================================================

print("\n============== NUMBA ==============")

pi_numba, numba_avg, numba_best = benchmark(
    montecarlo_numba,
    N
)


# ============================================================
# NUMBA SIGNATURE
# ============================================================

print("\n========= NUMBA SIGNATURE =========")

pi_sig, sig_avg, sig_best = benchmark(
    montecarlo_numba_signature,
    N
)


# ============================================================
# RESULTS
# ============================================================

print("\n================ FINAL RESULTS ================\n")

print(f"Reference PI : {np.pi}\n")

print("NumPy")
print(f"  Estimate     : {pi_numpy}")
print(f"  Error        : {abs(pi_numpy - np.pi)}")
print(f"  Average Time : {numpy_avg:.6f} s")
print(f"  Best Time    : {numpy_best:.6f} s\n")

print("Numba")
print(f"  Estimate     : {pi_numba}")
print(f"  Error        : {abs(pi_numba - np.pi)}")
print(f"  Average Time : {numba_avg:.6f} s")
print(f"  Best Time    : {numba_best:.6f} s\n")

print("Numba + Signature")
print(f"  Estimate     : {pi_sig}")
print(f"  Error        : {abs(pi_sig - np.pi)}")
print(f"  Average Time : {sig_avg:.6f} s")
print(f"  Best Time    : {sig_best:.6f} s\n")

Running warmup compilation...

============== NUMPY ==============
Run 1: 7.146422 s
Run 2: 3.679499 s
Run 3: 3.749613 s
Run 4: 2.848117 s
Run 5: 3.389213 s

============== NUMBA ==============
Run 1: 0.182846 s
Run 2: 0.143894 s
Run 3: 0.149514 s
Run 4: 0.174184 s
Run 5: 0.146637 s

========= NUMBA SIGNATURE =========
Run 1: 0.125488 s
Run 2: 0.132030 s
Run 3: 0.131066 s
Run 4: 0.139820 s
Run 5: 0.157271 s

================ FINAL RESULTS ================

Reference PI : 3.141592653589793

NumPy
  Estimate     : 3.14147144
  Error        : 0.00012121358979300112
  Average Time : 4.162573 s
  Best Time    : 2.848117 s

Numba
  Estimate     : 3.14175016
  Error        : 0.00015750641020684242
  Average Time : 0.159415 s
  Best Time    : 0.143894 s

Numba + Signature
  Estimate     : 3.14158292
  Error        : 9.733589793281539e-06
  Average Time : 0.137135 s
  Best Time    : 0.125488 s

